<a href="https://colab.research.google.com/github/NVHau-K14/Tuan04_ThucHanh_DeepLearning/blob/main/CNN_NhanDangKhuonMat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import os
import requests
import tarfile
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from PIL import Image

# ==============================================================================
# 1. TẢI VÀ GIẢI NÉN BỘ DỮ LIỆU LFW ĐÃ ĐƯỢC GÁN NHÃN GIỚI TÍNH
# ==============================================================================
dataset_url = "http://www.cs.columbia.edu/CAVE/databases/pubfig/download/lfw_attributes.txt"
# dataset_url = "https://figshare.com/articles/dataset/Labeled_Faces_in_the_Wild_with_Gender_and_Race_Labels/12028680"
# Tuy nhiên, tải bộ dữ liệu đầy đủ từ figshare có thể mất thời gian và cần tài khoản.

# Tạo dữ liệu giả lập cho mục đích minh họa
num_images_male = 1000
num_images_female = 1000
image_height, image_width = 128, 128

X = np.zeros((num_images_male + num_images_female, image_height, image_width, 3), dtype=np.uint8)
y = np.zeros(num_images_male + num_images_female, dtype=int)

# Tạo ảnh giả lập Nam
for i in range(num_images_male):
    img = Image.new('RGB', (image_width, image_height), color=(200, 200, 200))
    X[i] = np.array(img)
    y[i] = 0 # 0: Nam

# Tạo ảnh giả lập Nữ (ví dụ: có các vệt màu sáng đại diện cho tóc dài)
for i in range(num_images_female):
    img = Image.new('RGB', (image_width, image_height), color=(150, 150, 150))
    X[num_images_male + i] = np.array(img)
    y[num_images_male + i] = 1 # 1: Nữ

# ==============================================================================
# 2. TIỀN XỬ LÝ DỮ LIỆU
# ==============================================================================

# 1. Chia dữ liệu thành tập huấn luyện và tập kiểm tra (80% - 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_split=0.2, random_state=42)

# 2. Chuẩn hóa dữ liệu: Chuyển đổi giá trị pixel từ [0, 255] sang [0, 1]
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# 3. Mã hóa nhãn (One-Hot Encoding) cho 2 lớp: Nam, Nữ
y_train = to_categorical(y_train, num_classes=2)
y_test = to_categorical(y_test, num_classes=2)

# ==============================================================================
# 3. XÂY DỰNG MÔ HÌNH CNN
# ==============================================================================

model = Sequential()

# Lớp tích chập thứ 1: 32 bộ lọc, kích thước 3x3, hàm kích hoạt ReLU
model.add(Conv2D(32, (3, 3), activation='relu', input_shape=(image_height, image_width, 3)))
# Lớp MaxPooling thứ 1: kích thước 2x2
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lớp tích chập thứ 2: 64 bộ lọc, kích thước 3x3, hàm kích hoạt ReLU
model.add(Conv2D(64, (3, 3), activation='relu'))
# Lớp MaxPooling thứ 2: kích thước 2x2
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lớp tích chập thứ 3: 128 bộ lọc, kích thước 3x3, hàm kích hoạt ReLU
model.add(Conv2D(128, (3, 3), activation='relu'))
# Lớp MaxPooling thứ 3: kích thước 2x2
model.add(MaxPooling2D(pool_size=(2, 2)))

# Lớp làm phẳng: Chuyển dữ liệu 3D thành 1D
model.add(Flatten())

# Lớp kết nối đầy đủ (Fully Connected): 128 nút, hàm kích hoạt ReLU
model.add(Dense(128, activation='relu'))
# Lớp Dropout: Giảm overfitting bằng cách ngắt ngẫu nhiên 50% số nút
model.add(Dropout(0.5))

# Lớp đầu ra: 2 nút (cho 2 lớp), hàm kích hoạt Softmax để tính xác suất
model.add(Dense(2, activation='softmax'))

# Hiển thị tóm tắt kiến trúc mô hình
model.summary()

# ==============================================================================
# 4. BIÊN DỊCH VÀ HUẤN LUYỆN MÔ HÌNH
# ==============================================================================

# Biên dịch mô hình: Sử dụng optimizer Adam, hàm mất mát categorical_crossentropy
# và đánh giá độ chính xác (accuracy)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Tăng cường dữ liệu (Data Augmentation) để giảm overfitting
datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

# Huấn luyện mô hình
batch_size = 32
epochs = 10 # Số lượng epoch (vòng lặp qua toàn bộ dữ liệu)

history = model.fit(
    datagen.flow(X_train, y_train, batch_size=batch_size),
    steps_per_epoch=len(X_train) // batch_size,
    epochs=epochs,
    validation_data=(X_test, y_test)
)

# ==============================================================================
# 5. ĐÁNH GIÁ MÔ HÌNH
# ==============================================================================

# Đánh giá độ chính xác trên tập kiểm tra
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Độ chính xác trên tập kiểm tra: {accuracy * 100:.2f}%")

# Lưu mô hình đã huấn luyện
model.save("gender_recognition_model.h5")
print("Đã lưu mô hình: gender_recognition_model.h5")